In [2]:
from langchain_openai import OpenAI
from langchain_ollama import OllamaLLM
import prompts
import schemas
import json
from langchain_core.output_parsers import JsonOutputParser
import pandas as pd
from tqdm import tqdm
from langchain_core.prompts import PromptTemplate
import re
import glob
import os

In [3]:
with open("questions.json", "r") as f: 
    questions_json = json.load(f)

In [4]:
end_path = "data/results/"

In [5]:
reports_path = "data/ING"

In [6]:
report_paths = glob.glob(reports_path + "/*.json")

In [7]:
# Function to get a language model object (Ollama or ChatGPT)
def get_language_model(model_type: str = "ollama", model_name: str = "default-model"):
    """
    Returns a language model object (Ollama or ChatGPT).
    
    Args:
        model_type (str): The type of model to use ("ollama" or "chatgpt"). Default is "ollama".
        model_name (str): The name of the model to use. Default is "default-model".
    
    Returns:
        An instance of the selected language model.
    """
    if model_type.lower() == "ollama":
        return OllamaLLM(model=model_name)
    elif model_type.lower() == "chatgpt":
        return OpenAI(model=model_name)
    else:
        raise ValueError("Invalid model_type. Choose 'ollama' or 'chatgpt'.")

# Function to interact with thea language model
def query_llm(llm, question: str, page: str):
    """
    Queries the language model with a question and a page string.
    
    Args:
        llm: The language model object.
        question (str): The question to ask the model.
        page (str): The page string to provide context.
    
    Returns:
        str: The output of the language model.
    """
    prompt = f"Context: {page}\n\nQuestion: {question}"
    return llm.invoke(prompt)

In [8]:
llm = get_language_model(model_name="llama3:8b")
parser = JsonOutputParser()

In [9]:
# results = []

# for page in tqdm(report["pages"]): 
#     prompt = prompts.render_prompt(
#             questions_[0],
#             page["markdown"],
#             prompt_template_id="baseline",
#             few_shot=False,
#             truncate_at=None,
#         )

#     llm_response = llm.invoke(prompt.text)

#     match = re.search(r"\{.*\}", llm_response, re.DOTALL)
#     if not match:
#         print(llm_response)
#         #raise ValueError("No JSON found in LLM output!")
    

#     json_str = match.group(0)

#     parsed = parser.parse(json_str)

#     result_page = parsed
#     result_page["page"] = page["page"]
#     result_page["markdown"] = page["markdown"]
#     results.append(result_page)

# df = pd.DataFrame(results)

In [10]:
questions = [schemas.QuestionSpec(**q) for q in questions_json][2:3]
questions

[]

In [11]:
for report_path in tqdm(report_paths): 

    with open(report_path, "r") as f: 
        report = json.load(f)

    report_result_path = os.path.join(end_path, os.path.basename(report_path))
    os.makedirs(report_result_path, exist_ok =True)

    for question in questions: 
        results = []
        for page in tqdm(report["pages"]): 
            prompt = prompts.render_prompt(
                    question,
                    page["markdown"],
                    prompt_template_id="baseline",
                    few_shot=False,
                    truncate_at=None,
                )

            llm_response = llm.invoke(prompt.text)

            match = re.search(r"\{.*\}", llm_response, re.DOTALL)
            if match:
                json_str = match.group(0)

                parsed = parser.parse(json_str)

                result_page = parsed
                result_page["page"] = page["page"]
                result_page["markdown"] = page["markdown"]
                results.append(result_page)

        df = pd.DataFrame(results)
        df.to_csv(os.path.join(report_result_path, f"{question.id}.csv"))

0it [00:00, ?it/s]
